# Day D4 — 한 사람의 GLM: HRF convolution, 설계 행렬, contrast, 잡음과 자기상관

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/D-fmri/day-D4.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/D-fmri/day-D4.md)

In [ ]:
# Colab 기본 런타임에 없는 패키지 설치
%pip install -q nilearn

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 이중 감마 HRF와 블록 설계 회귀변수

In [ ]:
import numpy as np
from scipy.stats import gamma

def double_gamma(t, peak=6.0, under=16.0, ratio=1 / 6):
    """흔히 쓰는 모양의 이중 감마 HRF: 5초 근처 봉우리, 15–16초 근처 얕은 골"""
    h = gamma.pdf(t, peak) - ratio * gamma.pdf(t, under)
    return h / h.max()

dt = 0.1                                   # 고해상도 시간 격자(초)
t_hrf = np.arange(0, 32, dt)
hrf = double_gamma(t_hrf)
print("HRF 봉우리 시각: %.1f s | 가장 깊은 골: %.1f s" % (t_hrf[hrf.argmax()], t_hrf[hrf.argmin()]))

# 블록 설계: 20초 과제, 20초 쉼을 반복 (총 240초)
TR, total = 2.0, 240
t_hi = np.arange(0, total, dt)
boxcar = ((t_hi % 40) < 20).astype(float)
reg_hi = np.convolve(boxcar, hrf)[: t_hi.size] * dt          # 고해상도에서 convolution
regressor = reg_hi[:: int(TR / dt)]                            # TR 간격으로 뽑기
regressor /= regressor.max()

onset_lag = t_hi[np.argmax(reg_hi > 0.5 * reg_hi.max())]
print("과제는 0 s에 시작, 예측 BOLD가 절반 높이에 닿는 시각: %.1f s" % onset_lag)
print("회귀변수 길이:", regressor.size, "(= 240 s / TR 2 s)")
print("첫 12개 값:", np.round(regressor[:12], 2))

### 예시 2 — GLM 적합, contrast t, 그리고 자기상관을 무시했을 때의 대가

In [ ]:
import numpy as np
import statsmodels.api as sm
from scipy.stats import gamma

rng = np.random.default_rng(11)
TR, n = 2.0, 120
dt = 0.1
t_hrf = np.arange(0, 32, dt)
hrf = gamma.pdf(t_hrf, 6) - gamma.pdf(t_hrf, 16) / 6
t_hi = np.arange(0, n * TR, dt)

def make_regressor(onsets, dur):
    stim = np.zeros(t_hi.size)
    for o in onsets:
        stim[(t_hi >= o) & (t_hi < o + dur)] = 1
    r = (np.convolve(stim, hrf)[: t_hi.size] * dt)[:: int(TR / dt)]
    return r / r.max()

task = make_regressor(np.arange(0, n * TR, 40), 20)           # 20초 켜고 20초 끄기
tt = np.arange(n)
dct = np.column_stack([np.cos(np.pi * k * (tt + 0.5) / n) for k in range(1, 4)])  # 느린 흐름(drift)용 코사인 3개

def ar1_noise(phi, sd=1.0):
    e = np.zeros(n); e[0] = rng.normal(0, sd)
    for i in range(1, n):
        e[i] = phi * e[i - 1] + rng.normal(0, sd)
    return e

X = sm.add_constant(np.column_stack([task, dct]))             # 열: 절편, 과제, DCT 3개
c = np.array([0, 1, 0, 0, 0])                                 # contrast: '과제 효과'

# 한 voxel: 기저값 100, 과제 효과 2.0, 느린 drift, AR(1) 잡음
y = 100 + 2.0 * task + 1.5 * dct[:, 0] + ar1_noise(0.5)
ols = sm.OLS(y, X).fit()
print("OLS   β_task = %.2f, SE = %.3f, t = %.2f" % (ols.params[1], ols.bse[1], ols.tvalues[1]))
gls = sm.GLSAR(y, X, rho=1).iterative_fit(maxiter=5)          # 잔차로 AR(1)을 추정해 prewhitening
print("GLSAR β_task = %.2f, SE = %.3f, t = %.2f, 추정 ρ = %.2f" % (gls.params[1], gls.bse[1], gls.tvalues[1], gls.model.rho[0]))
print("contrast t (OLS) 직접 계산:", round(float(c @ ols.params / np.sqrt(c @ ols.cov_params() @ c)), 2))

# 신호가 전혀 없는 voxel 1,000개를 만들어 반복해 보자
b, se_ols, se_gls, p_ols, p_gls = [], [], [], [], []
for _ in range(1000):
    y0 = 100 + 1.5 * dct[:, 0] + ar1_noise(0.5)
    f1 = sm.OLS(y0, X).fit()
    f2 = sm.GLSAR(y0, X, rho=1).iterative_fit(maxiter=5)
    b.append(f1.params[1]); se_ols.append(f1.bse[1]); se_gls.append(f2.bse[1])
    p_ols.append(f1.pvalues[1]); p_gls.append(f2.pvalues[1])
print("β의 실제 흩어짐(SD): %.3f | OLS가 말하는 SE 평균: %.3f | GLSAR SE 평균: %.3f"
      % (np.std(b), np.mean(se_ols), np.mean(se_gls)))
print("거짓 양성 비율 — OLS: %.1f%%, GLSAR: %.1f%% (명목 5%%)"
      % (100 * np.mean(np.array(p_ols) < 0.05), 100 * np.mean(np.array(p_gls) < 0.05)))

설계 행렬을 그림으로 남기려면 예시 2 뒤에 아래를 붙입니다(그림 파일만 저장).

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

Path("figures/advanced/D").mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(4, 6))
ax.imshow((X - X.mean(0)) / (X.std(0) + 1e-12), aspect="auto", cmap="gray")
ax.set_xticks(range(X.shape[1])); ax.set_xticklabels(["const", "task", "dct1", "dct2", "dct3"], rotation=45)
ax.set_ylabel("volume (TR)")
fig.tight_layout(); fig.savefig("figures/advanced/D/D4_design_matrix.png", dpi=150)

### 예시 3 — 두 사건 사이 간격과 설계 효율

In [ ]:
import numpy as np
from scipy.stats import gamma

rng = np.random.default_rng(2)
TR, n, dt = 2.0, 150, 0.1
t_hrf = np.arange(0, 32, dt)
hrf = gamma.pdf(t_hrf, 6) - gamma.pdf(t_hrf, 16) / 6
t_hi = np.arange(0, n * TR, dt)

def reg(onsets, dur=1.0):
    s = np.zeros(t_hi.size)
    for o in onsets:
        s[(t_hi >= o) & (t_hi < o + dur)] = 1
    r = (np.convolve(s, hrf)[: t_hi.size] * dt)[:: int(TR / dt)]
    return r / r.max()

def efficiency(X, c):
    return 1 / (c @ np.linalg.inv(X.T @ X) @ c)    # 클수록 같은 잡음에서 추정이 덜 흔들린다

cue = np.cumsum(rng.uniform(12, 16, 18))            # '단서' 사건 18번
designs = {f"고정 {g} s": cue + g for g in [1, 2, 4]}
designs["무작위 1–8 s"] = cue + rng.uniform(1, 8, cue.size)
for name, target in designs.items():
    a, b = reg(cue), reg(target)                    # '단서'와 '목표' 회귀변수
    X = np.column_stack([np.ones(n), a, b])
    r = np.corrcoef(a, b)[0, 1]
    print(f"{name:12} 상관 {r:5.2f} | 효율(단서만) {efficiency(X, np.array([0, 1, 0])):5.1f}"
          f" | 효율(목표만) {efficiency(X, np.array([0, 0, 1])):5.1f}")

### 참고 — nilearn으로 같은 1수준 분석을 할 때의 형태 (nilearn 필요)

> 이 코드는 `pip install nibabel nilearn`으로 설치한 환경(nibabel 5.5, nilearn 0.14.1)에서, 같은 이름의 **가상 파일**을 만들어 그대로 실행해 오류 없이 도는 것을 확인했습니다. 출력은 여러분의 실제 파일에 따라 달라지므로 싣지 않았습니다.

In [ ]:
# nilearn이 필요하다: pip install nilearn
from nilearn.glm.first_level import FirstLevelModel
model = FirstLevelModel(t_r=2.0, hrf_model="spm", drift_model="cosine",
                        high_pass=0.01, noise_model="ar1", smoothing_fwhm=6)
model = model.fit("sub-01_task-x_bold.nii.gz", events=events_df, confounds=confounds_df)
z_map = model.compute_contrast("condA - condB", output_type="z_score")